# Notebook 1: ChEMBL Data Collection — GPCR Benchmark Panel

**Purpose:** Confirm ChEMBL target IDs and run a live volume/quality check for the
three **net-new** targets in the panel — ADORA2A, OPRM1, CCR5 — then fetch and save
their raw bioactivity records for downstream cleaning.

**DRD2 and CB2 are reused from existing project inputs
and are NOT fetched here.** This notebook covers only ADORA2A, OPRM1, and CCR5.

**Panel-lock gate:** the 5-target panel is not locked until this volume/quality
check runs against live ChEMBL. CCR5 in particular may need swapping for a
better-characterized alternative if its curated count or assay-annotation quality
is poor — the quality-check output below is what that call gets made from.

**OPRM1 note:** this notebook also pulls ChEMBL mechanism-of-action records
(`action_type`, e.g. agonist/antagonist) for OPRM1 only. This does NOT implement
function-aware splitting — that decision is still open — it only captures the data
needed to make that decision later without a second live ChEMBL round-trip.

In [ ]:
# MUST BE FIRST CELL!
import os
import multiprocessing

# ── Environment toggle ──
HPC_MODE = False  # True on HPC, False on Colab

if HPC_MODE:
    N_CORES = 22
    import matplotlib
    matplotlib.use('Agg')
else:
    N_CORES = min(multiprocessing.cpu_count(), 4)

for var in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
            'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS']:
    os.environ[var] = str(N_CORES)

os.environ['OMP_NESTED'] = 'FALSE'
os.environ['MKL_DYNAMIC'] = 'FALSE'

ENV = "HPC" if HPC_MODE else "Colab"
print(f"Environment: {ENV} | Using {N_CORES} cores")


In [ ]:
from pathlib import Path

if HPC_MODE:
    PROJECT_DIR = Path('./')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/My Drive/gpcr_benchmark')

# Create directory tree
for subdir in ['data/raw', 'data/processed', 'data/external/drugbank', 'data/quality',
               'ml/results', 'ml/models']:
    (PROJECT_DIR / subdir).mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")


In [ ]:
# RDKit / ChEMBL client
!pip install chembl_webresource_client -q


In [ ]:
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client


## Step 1 — Confirm ChEMBL target IDs (live, do not assume)

Search ChEMBL by name for each net-new target. Multiple hits are expected (species
variants, isoforms, chimeric constructs) — inspect the printed candidates and pick
the correct **human, single-protein** entry manually in Step 2. Do not auto-select
the first match.

In [ ]:
target_client = new_client.target
activity_client = new_client.activity
mechanism_client = new_client.mechanism

# Search terms per net-new target — broad on purpose, filter by eye below
search_terms = {
    'adora2a': ['Adenosine receptor A2a'],
    'oprm1':   ['Mu-type opioid receptor', 'Mu opioid receptor'],
    'ccr5':    ['C-C chemokine receptor type 5', 'CCR5'],
}

candidates = {}
for target_key, names in search_terms.items():
    print(f"\n=== Candidates for {target_key.upper()} ===")
    rows = []
    for name in names:
        hits = target_client.filter(pref_name__icontains=name).only(
            ['target_chembl_id', 'pref_name', 'organism', 'target_type']
        )
        rows.extend(list(hits))
    candidates[target_key] = rows
    if not rows:
        print("  No matches — widen search_terms for this target.")
    for r in rows:
        print(f"  {r['target_chembl_id']} | {r['pref_name']} | {r['organism']} | {r['target_type']}")


In [ ]:
# Full detail on each candidate — use this to confirm organism=Homo sapiens,
# target_type=SINGLE PROTEIN before locking IDs in the next cell.
for target_key, rows in candidates.items():
    print(f"\n=== {target_key.upper()} detail ===")
    for r in rows:
        info = target_client.get(r['target_chembl_id'])
        print(f"  {info['target_chembl_id']}: {info['pref_name']} "
              f"({info['organism']}, {info['target_type']}, "
              f"confidence={info.get('target_confidence')})")


## Step 2 — Lock confirmed IDs (manual, after inspecting Step 1 output)

Fill in the `target_chembl_id` for each target below from the candidates printed
above. Left as `None` on purpose — do not carry over a remembered/assumed ID.

In [ ]:
CONFIRMED_TARGETS = {
    'adora2a': 'CHEMBL251',  # Adenosine receptor A2a, Homo sapiens, SINGLE PROTEIN
    'oprm1':   'CHEMBL233',  # Mu-type opioid receptor, Homo sapiens, SINGLE PROTEIN
    'ccr5':    'CHEMBL274',  # C-C chemokine receptor type 5, Homo sapiens, SINGLE PROTEIN
}

missing = [k for k, v in CONFIRMED_TARGETS.items() if v is None]
assert not missing, f"Set target_chembl_id for: {missing} before continuing"


## Step 3 — Fetch activity records + live volume/quality check

For each target: pull `Ki`/`IC50`/`EC50` records (matches notebook 02's
`ACTIVITY_POOL` options), then compute the volume/quality metrics the panel-lock
decision depends on — raw count, standard-type breakdown, pChEMBL coverage, assay
count, and duplicate-measurement density (records per assay).

In [ ]:
import time
from IPython.display import display

raw_dfs = {}
quality_checks = {}
fetch_seconds = {}
overall_start = time.time()

for target_key, chembl_id in CONFIRMED_TARGETS.items():
    frozen_path = PROJECT_DIR / 'data' / 'raw' / f'raw_data_{target_key}.csv'
    if frozen_path.exists():
        # DIAGNOSTIC-COPY GUARD (added 2026-09-01): this file is already
        # frozen. Load it from disk instead of re-fetching live -- this
        # notebook's job in this copy is the relation diagnostic appended
        # below, not a fresh first-time collection, so there is no reason
        # to spend a live API call re-pulling data that will not be saved
        # (see the matching guard in the Step 5 save cell).
        print(f'{target_key.upper()}: frozen raw file found, loading from disk (not re-fetching live)')
        df_raw = pd.read_csv(frozen_path)
        raw_dfs[target_key] = df_raw
        quality_checks[target_key] = {'note': 'loaded from existing frozen file, not re-fetched'}
        fetch_seconds[target_key] = 0.0
        continue

    print(f"\n=== Fetching {target_key.upper()} ({chembl_id}) ===")
    t0 = time.time()

    # Unfiltered pull first — needed for the raw standard_type breakdown below
    all_activities = activity_client.filter(
        target_chembl_id=chembl_id,
        standard_value__isnull=False,
    ).only([
        'molecule_chembl_id', 'canonical_smiles', 'standard_type',
        'standard_value', 'standard_units', 'pchembl_value', 'assay_chembl_id'
    ])
    df_all = pd.DataFrame(all_activities)

    # Filtered to the three types notebook 02 actually pools
    df_raw = df_all[df_all['standard_type'].isin(['Ki', 'IC50', 'EC50'])].reset_index(drop=True)

    fetch_seconds[target_key] = time.time() - t0
    raw_dfs[target_key] = df_raw

    qc = {
        'n_raw_all_types': len(df_all),
        'n_raw_ki_ic50_ec50': len(df_raw),
        'standard_type_breakdown': df_all['standard_type'].value_counts().to_dict(),
        'pchembl_coverage_pct': round(100 * df_raw['pchembl_value'].notna().mean(), 1) if len(df_raw) else 0.0,
        'n_unique_assays': df_raw['assay_chembl_id'].nunique(),
        'n_unique_molecules': df_raw['molecule_chembl_id'].nunique(),
        'records_per_assay_median': (
            round(df_raw.groupby('assay_chembl_id').size().median(), 1) if len(df_raw) else 0.0
        ),
        'download_seconds': round(fetch_seconds[target_key], 1),
    }
    quality_checks[target_key] = qc

    print(f"  Raw (all types):        {qc['n_raw_all_types']}")
    print(f"  Raw (Ki/IC50/EC50):     {qc['n_raw_ki_ic50_ec50']}")
    print(f"  pChEMBL coverage:       {qc['pchembl_coverage_pct']}%")
    print(f"  Unique assays:          {qc['n_unique_assays']}")
    print(f"  Unique molecules:       {qc['n_unique_molecules']}")
    print(f"  Median records/assay:   {qc['records_per_assay_median']}")

    if qc['n_raw_ki_ic50_ec50'] < 500:
        print(f"  WARNING: {target_key.upper()} has < 500 usable records — "
              f"flag as panel-swap candidate before locking scope.")

print(f"\nTotal download time (3 targets): {time.time() - overall_start:.1f}s")

for tk in CONFIRMED_TARGETS:
    print(f'\n--- {tk.upper()} preview ---')
    display(raw_dfs[tk].head())


## Step 4 — OPRM1 mechanism-of-action pull (agonist/antagonist)

Captures `action_type` per molecule-target mechanism record for OPRM1 only, so the
open agonist/antagonist pooling question can be decided from data rather than
assumption — without implementing any splitting logic here.

In [ ]:
oprm1_id = CONFIRMED_TARGETS['oprm1']
mechanisms = mechanism_client.filter(target_chembl_id=oprm1_id).only(
    ['molecule_chembl_id', 'action_type', 'mechanism_of_action']
)
df_oprm1_mechanism = pd.DataFrame(mechanisms)

if len(df_oprm1_mechanism):
    action_type_breakdown = df_oprm1_mechanism['action_type'].value_counts().to_dict()
else:
    action_type_breakdown = {}

quality_checks['oprm1']['mechanism_action_type_breakdown'] = action_type_breakdown
quality_checks['oprm1']['n_mechanism_records'] = len(df_oprm1_mechanism)

print(f"OPRM1 mechanism records: {len(df_oprm1_mechanism)}")
print(f"action_type breakdown: {action_type_breakdown}")


## Step 5 — Save raw data, quality-check report, and manifest

In [ ]:
save_path = PROJECT_DIR / 'data' / 'raw'
quality_path = PROJECT_DIR / 'data' / 'quality'
save_path.mkdir(parents=True, exist_ok=True)
quality_path.mkdir(parents=True, exist_ok=True)

output_files = {}
for target_key, df_raw in raw_dfs.items():
    file_name = f'raw_data_{target_key}.csv'
    out_path = save_path / file_name
    if out_path.exists():
        # DIAGNOSTIC-COPY GUARD (added 2026-09-01): never overwrite an
        # already-frozen raw file. This notebook copy's purpose in its
        # current form is the read-only relation diagnostic appended below,
        # not a fresh first-time collection -- overwriting here would
        # silently replace the frozen dataset the rest of the project
        # (notebooks 02-07) already depends on.
        print(f'{target_key.upper()}: {out_path} already exists, NOT overwriting (diagnostic-copy guard)')
        output_files[file_name] = {'path': str(out_path), 'n_rows': len(df_raw), 'skipped_write': True}
        continue
    df_raw.to_csv(out_path, index=False)
    print(f'{target_key.upper()} raw dataset saved at {out_path} ({len(df_raw)} records)')
    output_files[file_name] = {'path': str(out_path), 'n_rows': len(df_raw)}

mechanism_path = save_path / 'oprm1_mechanism_of_action.csv'
if mechanism_path.exists():
    print(f'{mechanism_path} already exists, NOT overwriting (diagnostic-copy guard)')
    output_files['oprm1_mechanism_of_action.csv'] = {'path': str(mechanism_path), 'n_rows': len(df_oprm1_mechanism), 'skipped_write': True}
else:
    df_oprm1_mechanism.to_csv(mechanism_path, index=False)
    output_files['oprm1_mechanism_of_action.csv'] = {
        'path': str(mechanism_path), 'n_rows': len(df_oprm1_mechanism)
    }

import json as _json
qc_file = quality_path / 'quality_check_01_data_collection.json'
if qc_file.exists():
    print(f'{qc_file} already exists, NOT overwriting (diagnostic-copy guard)')
else:
    with open(qc_file, 'w') as f:
        _json.dump(quality_checks, f, indent=2, default=str)
    print(f'\nQuality-check report saved: {qc_file}')


def _capture_package_versions():
    packages = ['rdkit', 'numpy', 'pandas', 'sklearn', 'xgboost', 'lightgbm',
                'optuna', 'shap', 'joblib', 'crepes', 'scipy', 'meeko']
    versions = {}
    for pkg in packages:
        try:
            mod = __import__(pkg)
            versions[pkg] = getattr(mod, '__version__', 'unknown')
        except ImportError:
            pass
    return versions


def _sha256_of_file(path):
    """Reads a file already written to disk and hashes its actual bytes --
    never the manifest itself, which does not exist yet at the point this
    runs. Returns None (not an exception) if the path is missing, so one
    unreadable/optional output does not abort the whole manifest write."""
    import hashlib
    p = Path(path)
    if not p.exists():
        return None
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def write_manifest(manifest_path, config_summary, outputs):
    """
    Single source-of-truth record of what produced a given output file set:
    config used, output paths + row counts + SHA-256 hashes, timestamp, and
    (best-effort) the git commit hash if this repo is git-tracked.

    SHA-256 is computed here, once, from the actual bytes on disk after each
    output has already been written -- this must run AFTER every output
    file in `outputs` exists, and never hashes manifest_path itself (that
    file is what this call is about to create, so hashing it here would be
    circular and would not describe the diagnostic's real outputs anyway).
    """
    import datetime, subprocess, json as _json2
    git_hash = None
    try:
        git_hash = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], stderr=subprocess.DEVNULL, cwd=str(PROJECT_DIR)
        ).decode().strip()
    except Exception:
        pass  # not a git repo, or git unavailable — fine, best-effort only

    hashed_outputs = {}
    for name, meta in outputs.items():
        meta = dict(meta)
        if 'sha256' not in meta:
            meta['sha256'] = _sha256_of_file(meta.get('path'))
        hashed_outputs[name] = meta

    manifest = {
        'timestamp': datetime.datetime.now().isoformat(),
        'git_commit': git_hash,
        'config': config_summary,
        'outputs': hashed_outputs,
        'package_versions': _capture_package_versions(),
    }
    with open(manifest_path, 'w') as f:
        _json2.dump(manifest, f, indent=2, default=str)
    print(f'Manifest saved: {manifest_path}')
    return manifest


manifest_path_01 = save_path / 'manifest_01_data_collection.json'
if manifest_path_01.exists():
    print(f'{manifest_path_01} already exists, NOT overwriting (diagnostic-copy guard) '
          f'-- defining write_manifest() for later cells without re-writing this manifest.')
else:
    write_manifest(
        manifest_path_01,
        config_summary={'targets': CONFIRMED_TARGETS, 'quality_checks': quality_checks},
        outputs=output_files,
    )


## Step 6 — ChEMBL relation/censoring diagnostic

**Why this exists.** The original Step 3 pull above requests
`standard_value`, `standard_type`, `pchembl_value`, etc., but never requests
`standard_relation` — the ChEMBL field recording whether a measurement was
reported exactly (`=`) or as a censored/inequality-qualified bound (`<`,
`>`, `<=`, `>=`). None of `data/raw/raw_data_*.csv` for
any of the five targets have a relation column. This means the frozen
dataset cannot currently distinguish an exact measurement from a censored
one, and the curation pipeline (notebook 02) has never filtered on this.

**What this section does, and does not do.** It is a **read-only diagnostic
side-analysis**, not a recuration step:
- Cells 11 and 15 above (the original Step 3 fetch and Step 5 save) are
  now guarded in this diagnostic copy: Step 5 will never overwrite an
  already-frozen `data/raw/` file, and Step 3 loads from disk instead of
  re-fetching live when the frozen file already exists. This holds even if
  the whole notebook is run top-to-bottom.
- This section itself only re-queries ChEMBL for `standard_relation`,
  `activity_id`, and `canonical_smiles` (fields the original pull never
  requested), and joins the result back onto the already-frozen raw
  records. It does not write anywhere under `data/raw/` or
  `data/processed/` — its outputs go only to new files under
  `data/quality/`.
- It aggregates at the `molecule_chembl_id` level (row-level pActivity ->
  per-compound median), a **proxy** for notebook 02's final
  per-standardized-compound aggregation, not identical to it (notebook 02
  also runs RDKit standardization/deduplication first). Disclosed
  explicitly, not silently assumed — good enough to gauge magnitude, not a
  substitute for notebook 02 itself.

**Revision note (2026-09-01):** an earlier version of this section had three
problems, since fixed: (1) it relied on the user remembering to skip cells
11/15 rather than making the notebook itself safe to run top-to-bottom;
(2) unmatched/unresolved relation records were silently treated as "exact"
in the summary counts while being *excluded* from the exact-only comparison
— an inconsistency that could inflate the apparent label-change rate;
(3) the materiality verdict looked only at label changes among evaluable
compounds, ignoring the join match rate and the fraction of compounds with
zero exact-relation evidence at all (a compound entirely dropped under an
exact-only policy is a bigger effect than a label flip, and was not being
counted). The composite join key has also been widened to include
`standard_units` and `canonical_smiles`, since `molecule_chembl_id` +
`assay_chembl_id` + `standard_type` + `standard_value` alone can collide
when the same numeric value appears in different units within the same
assay/molecule/type combination.

**Revision note (2026-09-01, second pass):** `match_rate_pct` in the
previous revision was computed from `standard_relation` non-nullness alone,
which conflated two different things -- a composite-key match failing, and
a composite-key match succeeding while the relation field itself happened
to be null. The join cell now uses `merge(..., indicator=True)` to report
`key_match_rate_pct` (did the composite key match at all) and
`relation_resolved_rate_pct` (of the key-matched records, how many carry a
usable relation value) as two separate, correctly-named metrics. The
materiality verdict's evidence-coverage axis now uses
`relation_resolved_rate_pct` specifically, since that is the actual usable
evidence for an exact-only comparison; `key_match_rate_pct` is still
reported alongside for audit-traceability but is not itself a verdict
trigger.

**Revision note (2026-09-02, third pass -- final correction):** the first two
attempts at classifying zero-exact-evidence compounds were both wrong, in two
different ways. The first ignored censoring direction entirely (treated a
reported boundary as if it were the true value). The second accounted for
direction per record, but then took the single tightest floor and tightest
ceiling across ALL of a compound's censored records -- i.e., it asked whether
one latent value could satisfy every measurement simultaneously
(an intersection), which is not what the curation rule computes and produced
occasional internally-contradictory bounds (found in 4 CB2 and 3 OPRM1
compounds) because a `min`/`max` intersection has no guarantee of consistency
the way an order-statistic does.

The corrected approach propagates each record's directional pActivity
interval through the SAME per-compound median-aggregation rule the curation
pipeline actually uses, not an intersection: for a compound's set of n
censored/unresolved records, the minimum possible median is the median of
(each floor record's boundary, or -infinity for every ceiling/unresolved
record); the maximum possible median is the mirror image (each ceiling
record's boundary, or +infinity for every floor/unresolved record). A
compound is called `active` only if the minimum possible median already
clears the threshold, `inactive` only if the maximum possible median already
falls short of it, and `indeterminate` otherwise. `~` (approximate) and
unresolved-relation records contribute to neither bound (treated as fully
uninformative, i.e. -infinity in the minimum-median calculation and
+infinity in the maximum-median calculation), per the same disclosed
principle as the rest of this diagnostic: never assume a direction that
isn't actually known. `median_lower <= median_upper` is asserted for every
compound as a live consistency check (it is a mathematical necessity under
this construction, since both bracket the same achievable true median
pointwise) -- unlike the intersection approach, no compound can produce a
contradictory pair of bounds here.

This is still scoped as a `molecule_chembl_id`-level proxy for notebook 02's
final post-standardization compound identity (unchanged from the earlier
revisions' caveat), and still only covers the full Ki+IC50+EC50 record set
per compound (the full-pool measurement scope), not the ki-only or
ki_ic50 pools separately.

In [ ]:
# All five targets, not just the three net-new ones the original Step 3
# pulled — DRD2 and CB2 went through the same query shape (confirmed: their
# raw_data_*.csv files have the identical column set with no relation
# column either), so the diagnostic must cover all five to be complete.
ALL_TARGET_IDS = {
    'drd2':    'CHEMBL217',
    'cb2':     'CHEMBL253',
    'adora2a': 'CHEMBL251',
    'oprm1':   'CHEMBL233',
    'ccr5':    'CHEMBL274',
}

# Defensive re-init: makes this section runnable even if only cells 1-4 of
# the original notebook (environment + ChEMBL client) were executed, not
# the original Step 3 pull itself.
if 'activity_client' not in dir():
    from chembl_webresource_client.new_client import new_client
    activity_client = new_client.activity

print('Diagnostic targets:', list(ALL_TARGET_IDS))


In [ ]:
# Cached-input branch (added 2026-09-02): the query above hits ChEMBL, a
# live, versioned database -- a later run is not guaranteed to receive the
# same records as an earlier one, even with identical filter parameters.
# If the immutable diagnostic snapshot from a prior run already exists on
# disk, load it directly and skip the live relation query and join
# entirely, so every downstream cell (relation summary, label-change,
# median-interval classification, verdict) operates on the exact records
# this analysis was actually verified against, not on whatever ChEMBL
# happens to return today. Delete both cached files first if a genuinely
# fresh live pull is wanted.
_records_cache_path = PROJECT_DIR / 'data' / 'quality' / 'chembl_relation_censoring_diagnostic_records.csv'
_join_cache_path = PROJECT_DIR / 'data' / 'quality' / 'chembl_relation_join_report.csv'
_DIAGNOSTIC_CACHE_HIT = _records_cache_path.exists() and _join_cache_path.exists()

if _DIAGNOSTIC_CACHE_HIT:
    records_all = pd.read_csv(_records_cache_path)
    join_report_df = pd.read_csv(_join_cache_path)
    print(f'Cache hit: loaded {len(records_all)} records from {_records_cache_path.name} '
          f'and the join report from {_join_cache_path.name}.')
    print('Skipping the live ChEMBL relation re-query and join (next two cells) -- '
          'delete both cached files first if a fresh live pull is genuinely wanted.')
else:
    print('No cached diagnostic snapshot found at', _records_cache_path,
          '-- proceeding to the live relation re-query below.')


In [ ]:
# Targeted re-query: same filter shape as the original Step 3 pull
# (target_chembl_id, standard_value__isnull=False, standard_type in
# Ki/IC50/EC50), but this time requesting standard_relation, activity_id,
# and canonical_smiles too. activity_id is ChEMBL's own primary key for an
# activity record -- not present in the frozen raw files, so it cannot be
# used to join back directly this round, but it IS captured here so any
# future re-pull has a stable key instead of repeating a composite-key
# join. canonical_smiles strengthens the composite join key used below.
if not _DIAGNOSTIC_CACHE_HIT:
    relation_dfs = {}
    for target_key, chembl_id in ALL_TARGET_IDS.items():
        print(f'Fetching relation/activity_id for {target_key.upper()} ({chembl_id})...')
        activities = activity_client.filter(
            target_chembl_id=chembl_id,
            standard_value__isnull=False,
        ).only([
            'activity_id', 'molecule_chembl_id', 'canonical_smiles', 'assay_chembl_id',
            'standard_type', 'standard_value', 'standard_units', 'standard_relation',
        ])
        df_rel = pd.DataFrame(activities)
        df_rel = df_rel[df_rel['standard_type'].isin(['Ki', 'IC50', 'EC50'])].reset_index(drop=True)
        relation_dfs[target_key] = df_rel
        rel_counts = df_rel['standard_relation'].value_counts(dropna=False).to_dict()
        print(f'  {len(df_rel)} records | relation breakdown: {rel_counts}')
else:
    print('Skipped: using the cached diagnostic snapshot, no live ChEMBL relation query performed.')


In [ ]:
# Join the newly-fetched relation info onto the frozen raw records via a
# composite key. Widened to (molecule_chembl_id, assay_chembl_id,
# standard_type, standard_value, standard_units, canonical_smiles) --
# molecule/assay/type/value alone can collide if the same numeric value
# appears under different units within the same assay/molecule/type
# combination; units and the SMILES both narrow that collision risk
# further.
#
# Two DIFFERENT things are tracked and reported separately, not conflated
# into one "match rate": a composite-key match can succeed while the
# matched record's relation field still happens to be null. merge(...,
# indicator=True) gives an explicit, unambiguous key-match flag ('both' vs
# 'left_only') instead of inferring match success from standard_relation
# non-nullness, which cannot distinguish "no key match" from "matched, but
# relation itself was null."

JOIN_KEYS = ['molecule_chembl_id', 'assay_chembl_id', 'standard_type',
             'standard_value', 'standard_units', 'canonical_smiles']

if not _DIAGNOSTIC_CACHE_HIT:
    joined_dfs = {}
    join_reports = []
    for target_key in ALL_TARGET_IDS:
        raw_path = PROJECT_DIR / 'data' / 'raw' / f'raw_data_{target_key}.csv'
        df_frozen = pd.read_csv(raw_path)
        df_frozen['standard_value'] = pd.to_numeric(df_frozen['standard_value'], errors='coerce')

        df_rel = relation_dfs[target_key].copy()
        df_rel['standard_value'] = pd.to_numeric(df_rel['standard_value'], errors='coerce')
        # Keep first match per key -- a small number of true duplicate keys are
        # still possible (rare exact replicate rows); acceptable for a
        # magnitude diagnostic, not for recuration.
        df_rel_dedup = df_rel.drop_duplicates(subset=JOIN_KEYS, keep='first')

        merged = df_frozen.merge(
            df_rel_dedup[JOIN_KEYS + ['standard_relation', 'activity_id']],
            on=JOIN_KEYS, how='left', indicator=True,
        )
        key_matched = merged['_merge'] == 'both'
        n_key_matched = int(key_matched.sum())
        n_total = len(merged)

        relation_resolved = key_matched & merged['standard_relation'].notna()
        n_relation_resolved = int(relation_resolved.sum())

        joined_dfs[target_key] = merged.drop(columns='_merge')
        join_reports.append({
            'target': target_key,
            'n_frozen_records': n_total,
            'n_key_matched': n_key_matched,
            'key_match_rate_pct': round(100 * n_key_matched / n_total, 2) if n_total else 0.0,
            'n_relation_resolved': n_relation_resolved,
            # Denominator is key-matched records, not all records -- this is the
            # fraction of the USABLE (key-matched) evidence that actually has a
            # relation value, which is what an exact-only analysis can draw on.
            'relation_resolved_rate_pct': (
                round(100 * n_relation_resolved / n_key_matched, 2) if n_key_matched else 0.0
            ),
        })
        print(f"{target_key.upper()}: key-matched {n_key_matched}/{n_total} "
              f"({100 * n_key_matched / n_total:.1f}%) | of those, relation resolved "
              f"{n_relation_resolved}/{n_key_matched} "
              f"({100 * n_relation_resolved / n_key_matched if n_key_matched else 0:.1f}%)")

    join_report_df = pd.DataFrame(join_reports)
    print()
    print(join_report_df.to_string(index=False))
else:
    print('Skipped: using the cached join_report_df (see cache-check cell above).')
    print(join_report_df.to_string(index=False))


In [ ]:
# Row-level pActivity, replicating notebook 02's own conversion exactly
# (Step 2, convert_and_validate_data): prefer pchembl_value when present,
# otherwise compute -log10(value_in_M) from standard_value/standard_units.
# ACTIVITY_THRESHOLD matches notebook 02's ACTIVITY_THRESHOLD = 6.0 exactly
# -- this must stay a single shared constant, not redefined independently.
#
# relation_status is a three-state category, resolved once and used
# consistently everywhere below -- this replaces an earlier version's
# boolean is_censored flag, which silently folded "unresolved" into
# "not censored" in the summary counts while a separate cell excluded
# unresolved rows from the exact-only comparison. Unresolved records are
# never treated as exact, and never treated as censored -- they are their
# own reported category throughout.
ACTIVITY_THRESHOLD = 6.0
BORDERLINE_MARGIN = 0.3  # log10 units either side of the threshold -- a
                          # disclosed, adjustable choice, not a hidden default

def _to_nm(value, unit):
    if pd.isna(value) or value <= 0:
        return float('nan')
    return value * {'nM': 1, 'uM': 1e3, 'mM': 1e6, 'M': 1e9}.get(unit, float('nan'))

def _relation_status(rel):
    if pd.isna(rel):
        return 'unresolved'
    return 'exact' if rel == '=' else 'censored'

# ACTIVITY_THRESHOLD/BORDERLINE_MARGIN are defined above unconditionally;
# is_borderline_censored is computed once, further below, AFTER this
# if/else -- unconditionally and only once -- so it exists identically
# whether records_all was just loaded from cache or freshly built here.
if not _DIAGNOSTIC_CACHE_HIT:
    record_rows = []
    for target_key, df in joined_dfs.items():
        d = df.copy()
        d['standard_value_nm'] = d.apply(lambda r: _to_nm(r['standard_value'], r['standard_units']), axis=1)
        d['pActivity_calculated'] = -np.log10(d['standard_value_nm'] * 1e-9)
        d['pActivity_row'] = pd.to_numeric(d['pchembl_value'], errors='coerce').fillna(d['pActivity_calculated'])
        d = d.replace([np.inf, -np.inf], np.nan).dropna(subset=['pActivity_row'])
        d = d[d['pActivity_row'].between(0, 14)]
        d['target'] = target_key
        d['relation_status'] = d['standard_relation'].apply(_relation_status)
        record_rows.append(d)
    records_all = pd.concat(record_rows, ignore_index=True)
else:
    print('Skipped: row-level pActivity/relation_status already present in the cached records.')

# Computed exactly once, here, regardless of which branch above ran -- a
# pure function of columns present either way (pActivity_row,
# relation_status). Must not live inside the "fresh run" branch only, or a
# cache-hit run would never populate it and the relation-summary cell
# below (which still reports it descriptively) would KeyError.
records_all['is_borderline_censored'] = (
    (records_all['relation_status'] == 'censored')
    & ((records_all['pActivity_row'] - ACTIVITY_THRESHOLD).abs() <= BORDERLINE_MARGIN)
)

status_counts = records_all['relation_status'].value_counts()
print(f'Total row-level records with a usable pActivity: {len(records_all)}')
print('Relation-status breakdown (exact / censored / unresolved), all targets combined:')
print(status_counts.to_string())


In [ ]:
# Per-target relation breakdown and borderline-censoring counts. Exact,
# censored, and unresolved are reported as three separate, non-overlapping
# categories -- unresolved is never folded into either of the other two.
relation_summary_rows = []
for target_key, d in records_all.groupby('target'):
    status_counts = d['relation_status'].value_counts()
    n_censored = int((d['relation_status'] == 'censored').sum())
    n_unresolved = int((d['relation_status'] == 'unresolved').sum())
    n_borderline = int(d['is_borderline_censored'].sum())
    relation_summary_rows.append({
        'target': target_key,
        'n_records': len(d),
        'n_exact': int((d['relation_status'] == 'exact').sum()),
        'n_censored': n_censored,
        'n_unresolved': n_unresolved,
        'pct_censored_of_all': round(100 * n_censored / len(d), 2) if len(d) else 0.0,
        'pct_unresolved_of_all': round(100 * n_unresolved / len(d), 2) if len(d) else 0.0,
        'n_borderline_censored': n_borderline,
        'pct_borderline_censored_of_all': round(100 * n_borderline / len(d), 3) if len(d) else 0.0,
        'relation_value_counts': d['standard_relation'].value_counts(dropna=False).to_dict(),
    })

relation_summary_df = pd.DataFrame(relation_summary_rows)
print(relation_summary_df.drop(columns='relation_value_counts').to_string(index=False))
print()
for row in relation_summary_rows:
    print(f"{row['target'].upper()} relation breakdown: {row['relation_value_counts']}")


In [ ]:
# The decision-relevant number: per compound (molecule_chembl_id, proxy for
# the final standardized compound -- see Step 6 markdown caveat), compute
# the median pActivity (a) using ALL matched-or-unresolved records, same as
# the current frozen pipeline's behavior (it does not distinguish relation
# at all), and (b) using ONLY confirmed-exact records, dropping anything
# censored OR unresolved. Compare the resulting active/inactive label at
# ACTIVITY_THRESHOLD between (a) and (b). This part is unchanged from the
# earlier revision and remains valid -- it never interprets a censored
# record's boundary as a point estimate, it only includes/excludes whole
# records from a median, which is direction-agnostic by construction.
label_change_rows = []
for target_key, d in records_all.groupby('target'):
    per_compound_all = d.groupby('molecule_chembl_id')['pActivity_row'].median()
    exact_only = d[d['relation_status'] == 'exact']
    per_compound_exact = exact_only.groupby('molecule_chembl_id')['pActivity_row'].median()

    compare = pd.DataFrame({'median_all': per_compound_all, 'median_exact_only': per_compound_exact})
    n_no_exact_evidence = int(compare['median_exact_only'].isna().sum())
    n_compounds = len(compare)

    evaluable = compare.dropna(subset=['median_exact_only']).copy()
    evaluable['label_all'] = evaluable['median_all'] >= ACTIVITY_THRESHOLD
    evaluable['label_exact_only'] = evaluable['median_exact_only'] >= ACTIVITY_THRESHOLD
    n_label_changed = int((evaluable['label_all'] != evaluable['label_exact_only']).sum())

    label_change_rows.append({
        'target': target_key,
        'n_compounds_total': n_compounds,
        'n_compounds_zero_exact_evidence': n_no_exact_evidence,
        'pct_compounds_zero_exact_evidence': round(100 * n_no_exact_evidence / n_compounds, 3) if n_compounds else 0.0,
        'n_compounds_evaluable': len(evaluable),
        'n_compounds_label_would_change': n_label_changed,
        'pct_compounds_label_would_change_of_evaluable': (
            round(100 * n_label_changed / len(evaluable), 3) if len(evaluable) else 0.0
        ),
    })

label_change_df = pd.DataFrame(label_change_rows)
print(label_change_df.to_string(index=False))


In [ ]:
# Corrected classification for zero-exact-evidence compounds: propagate each
# record's directional pActivity interval through the SAME per-compound
# median-aggregation rule the curation pipeline actually uses (order
# statistics), not an intersection of all bounds. See the Step 6 markdown
# revision note (third pass) for the full derivation and why the two earlier
# attempts (direction-blind, then intersection-based) were both wrong.
INF = float('inf')

def _bound_type(row):
    if row['relation_status'] != 'censored':
        return 'uninformative'   # unresolved, or anything not censored/exact
    rel = row['standard_relation']
    if rel in ('<', '<='):
        return 'floor'      # true pActivity >= boundary, unbounded above
    if rel in ('>', '>='):
        return 'ceiling'    # true pActivity <= boundary, unbounded below
    return 'uninformative'  # '~' and any other non-directional relation

records_all['bound_type'] = records_all.apply(_bound_type, axis=1)

def _classify_compound(g):
    floors = g.loc[g['bound_type'] == 'floor', 'pActivity_row'].tolist()
    ceilings = g.loc[g['bound_type'] == 'ceiling', 'pActivity_row'].tolist()
    n_uninformative = int((g['bound_type'] == 'uninformative').sum())
    n = len(g)

    lower_values = floors + [-INF] * (len(ceilings) + n_uninformative)
    upper_values = ceilings + [INF] * (len(floors) + n_uninformative)
    assert len(lower_values) == n and len(upper_values) == n

    median_lower = float(np.median(lower_values))
    median_upper = float(np.median(upper_values))
    # Mathematical necessity, not an assumption: both bracket the same
    # achievable true median pointwise, so median_lower <= median_upper
    # always holds. Checked live against real (messy) data rather than
    # trusted -- this is exactly the consistency check the intersection-
    # based approach was missing.
    assert median_lower <= median_upper, (g.name, median_lower, median_upper)

    if median_lower >= ACTIVITY_THRESHOLD:
        call = 'active'
    elif median_upper < ACTIVITY_THRESHOLD:
        call = 'inactive'
    else:
        call = 'indeterminate'
    return pd.Series({'call': call, 'median_lower_bound': median_lower, 'median_upper_bound': median_upper})

median_interval_rows = []
for target_key, d in records_all.groupby('target'):
    exact_ids = set(d.loc[d['relation_status'] == 'exact', 'molecule_chembl_id'])
    zero_exact = d[~d['molecule_chembl_id'].isin(exact_ids)]
    n_all_compounds = d['molecule_chembl_id'].nunique()

    per_compound = zero_exact.groupby('molecule_chembl_id').apply(_classify_compound, include_groups=False)
    counts = per_compound['call'].value_counts() if len(per_compound) else pd.Series(dtype=int)
    n_indeterminate = int(counts.get('indeterminate', 0))

    median_interval_rows.append({
        'target': target_key,
        'n_all_compounds': n_all_compounds,
        'n_zero_exact_compounds': len(per_compound),
        'n_active_by_median_interval': int(counts.get('active', 0)),
        'n_inactive_by_median_interval': int(counts.get('inactive', 0)),
        'n_indeterminate_by_median_interval': n_indeterminate,
        'pct_indeterminate_of_all_compounds': (
            round(100 * n_indeterminate / n_all_compounds, 3) if n_all_compounds else 0.0
        ),
    })

median_interval_df = pd.DataFrame(median_interval_rows)
print(median_interval_df.to_string(index=False))


In [ ]:
# Verdict -- a genuine composite across three independent axes, each with
# its own disclosed threshold:
#   (a) relation_resolved_rate_pct -- of the records that DID key-match,
#       what fraction actually carry a usable relation value. key_match_rate_pct
#       is still reported alongside for audit-traceability but is not itself
#       a materiality trigger.
#   (b) pct_indeterminate_of_all_compounds -- the corrected, direction- and
#       median-aware fraction of compounds whose censored-only evidence
#       cannot establish which side of the threshold their median pActivity
#       falls on (see the median-interval classification cell above). This
#       REPLACES the earlier, cruder "zero exact evidence" percentage, which
#       described data sparsity, not label uncertainty, and overstated risk
#       by lumping in compounds whose censored bounds actually resolve
#       cleanly once propagated correctly.
#   (c) label-change rate among evaluable compounds -- unchanged, direction-
#       agnostic by construction, still valid from the first revision.
# The reported verdict reflects whichever axis is worst, named explicitly.
RELATION_RESOLVED_MIN_PCT = 95.0
INDETERMINATE_MAX_PCT = 5.0
LABEL_CHANGE_MAX_PCT = 1.0

print(f'Materiality rule (any one triggers MATERIAL):')
print(f'  relation_resolved_rate_pct < {RELATION_RESOLVED_MIN_PCT}%')
print(f'  OR pct_indeterminate_of_all_compounds > {INDETERMINATE_MAX_PCT}%')
print(f'  OR label changes among evaluable compounds > {LABEL_CHANGE_MAX_PCT}%\n')

join_lookup = join_report_df.set_index('target')[
    ['key_match_rate_pct', 'relation_resolved_rate_pct']
].to_dict('index')
median_lookup = median_interval_df.set_index('target')['pct_indeterminate_of_all_compounds'].to_dict()

verdicts = []
for row in label_change_rows:
    t = row['target']
    key_match_rate = join_lookup.get(t, {}).get('key_match_rate_pct', 0.0)
    relation_resolved_rate = join_lookup.get(t, {}).get('relation_resolved_rate_pct', 0.0)
    indeterminate_pct = median_lookup.get(t, 0.0)
    label_change_pct = row['pct_compounds_label_would_change_of_evaluable']

    triggers = []
    if relation_resolved_rate < RELATION_RESOLVED_MIN_PCT:
        triggers.append(f'relation_resolved={relation_resolved_rate:.1f}%<{RELATION_RESOLVED_MIN_PCT}%')
    if indeterminate_pct > INDETERMINATE_MAX_PCT:
        triggers.append(f'indeterminate={indeterminate_pct:.2f}%>{INDETERMINATE_MAX_PCT}%')
    if label_change_pct > LABEL_CHANGE_MAX_PCT:
        triggers.append(f'label_change={label_change_pct:.3f}%>{LABEL_CHANGE_MAX_PCT}%')

    verdict = 'MATERIAL' if triggers else 'negligible'
    verdicts.append({
        **row,
        'key_match_rate_pct': key_match_rate,
        'relation_resolved_rate_pct': relation_resolved_rate,
        'pct_indeterminate_of_all_compounds': indeterminate_pct,
        'verdict': verdict,
        'triggered_by': '; '.join(triggers) if triggers else 'none',
    })
    trigger_note = f" ({verdicts[-1]['triggered_by']})" if triggers else ''
    print(f"{t.upper():8s} -> {verdict}{trigger_note}  "
          f"[key_match={key_match_rate:.1f}%, relation_resolved={relation_resolved_rate:.1f}%, "
          f"indeterminate={indeterminate_pct:.2f}%]")

verdict_df = pd.DataFrame(verdicts)


In [ ]:
# Save -- new files under data/quality/ only. Nothing in data/raw/ or
# data/processed/ is written or modified by this section (and, per the
# guards added to cells 11/15 above, nothing in data/raw/ can be
# overwritten by this notebook copy at all, regardless of run order).
quality_path = PROJECT_DIR / 'data' / 'quality'
quality_path.mkdir(parents=True, exist_ok=True)

records_out_path = quality_path / 'chembl_relation_censoring_diagnostic_records.csv'
records_all.drop(columns=['pActivity_calculated', 'is_borderline_censored'], errors='ignore').to_csv(records_out_path, index=False)  # errors='ignore': a cache-hit run's records_all may already lack these columns; bound_type is retained as informative per-record classification metadata

join_report_path = quality_path / 'chembl_relation_join_report.csv'
join_report_df.to_csv(join_report_path, index=False)

relation_summary_path = quality_path / 'chembl_relation_censoring_summary.csv'
relation_summary_df.drop(columns='relation_value_counts').to_csv(relation_summary_path, index=False)

median_interval_path = quality_path / 'chembl_relation_median_interval_classification.csv'
median_interval_df.to_csv(median_interval_path, index=False)

label_change_path = quality_path / 'chembl_relation_label_change_summary.csv'
verdict_df.to_csv(label_change_path, index=False)

print(f'wrote {records_out_path} ({len(records_all)} rows)')
print(f'wrote {join_report_path}')
print(f'wrote {relation_summary_path}')
print(f'wrote {median_interval_path}')
print(f'wrote {label_change_path}')

diagnostic_manifest_path = quality_path / 'manifest_chembl_relation_diagnostic.json'
write_manifest(
    diagnostic_manifest_path,
    config_summary={
        'purpose': 'ChEMBL standard_relation/censoring diagnostic -- read-only side analysis, '
                   'does not modify data/raw/ or data/processed/',
        'targets': ALL_TARGET_IDS,
        'activity_threshold': ACTIVITY_THRESHOLD,
        'materiality_rule': {
            'relation_resolved_min_pct': RELATION_RESOLVED_MIN_PCT,
            'indeterminate_max_pct': INDETERMINATE_MAX_PCT,
            'label_change_max_pct': LABEL_CHANGE_MAX_PCT,
        },
        'join_keys': JOIN_KEYS,
        'relation_status_categories': ['exact', 'censored', 'unresolved'],
        'indeterminate_classification_method': (
            "median-interval propagation: for zero-exact-evidence compounds, each censored "
            "record contributes a directional pActivity bound (floor for '<'/'<=', ceiling for "
            "'>'/'>='), '~' and unresolved relations contribute no bound in either direction; "
            "a compound is 'active' only if the minimum possible median already clears "
            "ACTIVITY_THRESHOLD, 'inactive' only if the maximum possible median already falls "
            "short of it, else 'indeterminate'. Supersedes two earlier, incorrect attempts "
            "(direction-blind median-of-reported-boundary, then intersection-of-all-bounds) -- "
            "see Step 6 markdown revision notes."
        ),
        'aggregation_note': "per-molecule_chembl_id median, a proxy for notebook 02's final "
                             "per-standardized-compound median -- see Step 6 markdown caveat",
    },
    outputs={
        'chembl_relation_censoring_diagnostic_records.csv': {'path': str(records_out_path), 'n_rows': len(records_all)},
        'chembl_relation_join_report.csv': {'path': str(join_report_path), 'n_rows': len(join_report_df)},
        'chembl_relation_censoring_summary.csv': {'path': str(relation_summary_path), 'n_rows': len(relation_summary_df)},
        'chembl_relation_median_interval_classification.csv': {'path': str(median_interval_path), 'n_rows': len(median_interval_df)},
        'chembl_relation_label_change_summary.csv': {'path': str(label_change_path), 'n_rows': len(verdict_df)},
    },
)


**Reading the output.** `chembl_relation_median_interval_classification.csv`
carries the corrected, direction-and-median-aware indeterminate fraction per
target -- the number that answers "for compounds with no exact measurement,
can the median-based label be confirmed from the censoring bounds alone."
`chembl_relation_label_change_summary.csv` carries the composite verdict:
relation-resolution coverage, this corrected indeterminate fraction, and the
label-change rate among evaluable compounds, with `triggered_by` stating
which axis (if any) is MATERIAL. If every target comes back negligible on
all three axes, the Methods disclosure can state a fully quantified, checked
result. If any target is MATERIAL, that target's compound-level detail is in
`chembl_relation_censoring_diagnostic_records.csv` for closer inspection --
deciding whether to act on that is a separate decision from having run this
diagnostic.